In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

np.random.seed(42)

# Generate binary classification dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    random_state=42
)

y = y.reshape(-1, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class distribution:", np.bincount(y.flatten()))

X shape: (1000, 10)
y shape: (1000, 1)
Class distribution: [498 502]


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (800, 10)
Test shape: (200, 10)


In [3]:
class LogisticRegressionGD:
    def __init__(self, learning_rate=0.01, epochs=1000, verbose=True, verbose_every=100):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.verbose = verbose
        self.verbose_every = verbose_every

        self.weights = None
        self.bias = None
        self.cost_history = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        m, n = X.shape

        self.weights = np.random.randn(n, 1) * 0.01
        self.bias = np.zeros((1,1))

        if self.verbose:
            print(f"Logistic Regression Training:")
            print(f" X shape: {X.shape}")
            print(f" y shape: {y.shape}")
            print(f" weights shape: {self.weights.shape}")
            print("-" * 50)

        for epoch in range(self.epochs):

            # Forward Pass
            z = X.dot(self.weights) + self.bias # (m, 1)

            # Sigmoid activation - converts to probability
            y_hat = self._sigmoid(z)            # (m, 1)


            # Compute Cost
            y_hat_clipped = np.clip(y_hat, 1e-7, 1 - 1e-7)
            cost = -(1/m) * np.sum(
                y * np.log(y_hat_clipped) + 
                (1 - y) * np.log(1 - y_hat_clipped)
            )
            self.cost_history.append(cost)
            
            # COMPUTE GRADIENTS 
            # Identical to Linear Regression — just errors are different
            errors = y_hat - y                    # (m, 1)
            dW = (1/m) * X.T.dot(errors)          # (n, 1)
            db = (1/m) * np.sum(errors)            # scalar
            
            # UPDATE PARAMETERS 
            self.weights = self.weights - self.learning_rate * dW
            self.bias = self.bias - self.learning_rate * db
            
            if self.verbose and epoch % self.verbose_every == 0:
                print(f"Epoch {epoch:4d}/{self.epochs} | "
                      f"Cost: {cost:.6f}")
        
        if self.verbose:
            print("-" * 50)
            print(f"Training complete!")
            print(f"Final weights: {self.weights.flatten().round(4)}")
        
        return self
    
    def predict_proba(self, X):
        z = X.dot(self.weights) + self.bias
        return self._sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        predictions = self.predict(X)
        return np.mean(predictions == y)
    
    def plot_cost_history(self):
        plt.figure(figsize=(10, 5))
        plt.plot(range(self.epochs), self.cost_history, 'b-')
        plt.xlabel('Epoch')
        plt.ylabel('Binary Cross Entropy')
        plt.title('Cost History — Logistic Regression')
        plt.grid(True)
        plt.show()

In [4]:
model = LogisticRegressionGD(
    learning_rate=0.1,
    epochs=1000,
    verbose=True,
    verbose_every=100
)

model.fit(X_train_scaled, y_train)

Logistic Regression Training:
 X shape: (800, 10)
 y shape: (800, 1)
 weights shape: (10, 1)
--------------------------------------------------
Epoch    0/1000 | Cost: 0.691607
Epoch  100/1000 | Cost: 0.396433
Epoch  200/1000 | Cost: 0.388057
Epoch  300/1000 | Cost: 0.386691
Epoch  400/1000 | Cost: 0.386387
Epoch  500/1000 | Cost: 0.386311
Epoch  600/1000 | Cost: 0.386291
Epoch  700/1000 | Cost: 0.386286
Epoch  800/1000 | Cost: 0.386284
Epoch  900/1000 | Cost: 0.386284
--------------------------------------------------
Training complete!
Final weights: [ 0.055   0.4517  0.0263  0.1157  0.5759 -1.2811  0.1318  0.0354  0.1592
  1.1093]


In [5]:
# Predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)

# Accuracy
print(f"Our Model Accuracy: {model.score(X_test_scaled, y_test):.4f}")

# Compare with sklearn
sk_model = LogisticRegression(random_state=42)
sk_model.fit(X_train_scaled, y_train.ravel())
print(f"Sklearn Accuracy:   {sk_model.score(X_test_scaled, y_test.ravel()):.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ROC-AUC
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

Our Model Accuracy: 0.8200
Sklearn Accuracy:   0.8200

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.77      0.81       100
           1       0.79      0.87      0.83       100

    accuracy                           0.82       200
   macro avg       0.82      0.82      0.82       200
weighted avg       0.82      0.82      0.82       200

ROC-AUC Score: 0.8797
